# HuggingFace Model Migration

This notebook shows how to migrate your Lightning checkpoint to a HuggingFace-compatible model format that decouples code from checkpoints.

## Why Migrate?

1. **Code-Checkpoint Decoupling**: Load models without maintaining source code
2. **Easy Sharing**: Push to HuggingFace Hub with one command
3. **Standard API**: Use `AutoModel.from_pretrained()` just like any other HF model
4. **Future-Proof**: Update code without affecting model loading

## Workflow Overview

1. Setup & Load Checkpoint
2. Extract Model Parameters
3. Create HuggingFace Model Directory
4. Save Model Code & Weights
5. Test Loading Locally
6. Verify Outputs Match Original
7. (Optional) Push to HuggingFace Hub

## Step 1: Setup - Load Your Current Checkpoint

In [3]:
import os
os.chdir("../")

import sys
import torch
import json
import shutil
from pathlib import Path

In [2]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Point to your checkpoint - modify these paths based on your setup
# SNGP Models:
ckpt_path = "checkpoints/acevedo_sngp_resnet18/model.ckpt"
model_type = "sngp"  # Set to "sngp" or "baseline"

# Baseline Models (uncomment to use instead):
# ckpt_path = "checkpoints/acevedo_baseline_resnet18/model.ckpt"
# model_type = "baseline"

# Load checkpoint
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f"✓ Loaded checkpoint from: {ckpt_path}")
print(f"  Checkpoint keys: {checkpoint.keys()}")
print(f"  Model type: {model_type}")

In [ ]:
# Inspect state dict structure
state_dict = checkpoint['state_dict']
print(f"Number of parameters: {len(state_dict)}")
print(f"\nFirst 5 keys:")
for i, key in enumerate(list(state_dict.keys())[:5]):
    print(f"  {i+1}. {key}")

## Step 2: Extract Model Parameters from Checkpoint

The Lightning checkpoint has keys like `net.feature_extractor.layer1...` because they're wrapped. We need to remove the `net.` prefix and skip parametrization buffers (for SNGP models).

In [ ]:
def extract_state_dict(checkpoint, device):
    """
    Extract and remap state dict from Lightning checkpoint.
    
    - Removes 'net.' prefix (Lightning module wrapper)
    - Skips parametrization buffers (PyTorch spectral norm internals)
    - Keeps weight_orig, weight_u, weight_v (spectral norm components for SNGP)
    """
    state_dict = checkpoint['state_dict']
    
    remapped_state_dict = {}
    skipped_keys = []
    
    for key, value in state_dict.items():
        # Remove 'net.' prefix (Lightning module wrapper)
        new_key = key.replace('net.', '', 1)
        
        # Skip parametrization buffers created by PyTorch's spectral_norm wrapper
        if 'parametrizations' in new_key:
            skipped_keys.append(new_key)
            continue
        
        remapped_state_dict[new_key] = value
    
    if skipped_keys:
        print(f"⚠ Skipped {len(skipped_keys)} parametrization buffer keys")
    
    return remapped_state_dict

# Extract the clean state dict
clean_state_dict = extract_state_dict(checkpoint, device)
print(f"\n✓ Extracted state dict: {len(clean_state_dict)} keys")

## Step 3: Create HuggingFace Model Directory Structure

In [ ]:
# Configure model parameters based on type
if model_type == "sngp":
    model_config = {
        "arch": "resnet18",
        "num_classes": 8,
        "rff_dim": 1024,
        "length_scale": 1.0,
        "ridge_penalty": 1e-3,
        "cov_momentum": 0.999,
        "mean_field": True,
        "n_power_iterations_sn": 1,
        "pretrained": False,
    }
    architectures = "SNGPForImageClassification"
    model_type_str = "sngp_classifier"
    hf_model_file = "hf_sngp_model"
else:
    model_config = {
        "arch": "resnet18",
        "num_classes": 8,
        "dropout_p": 0.5,
        "pretrained": False,
    }
    architectures = "BaselineClassifierForImageClassification"
    model_type_str = "baseline_classifier"
    hf_model_file = "hf_model"

print(f"Model config for {model_type}:")
for k, v in model_config.items():
    print(f"  {k}: {v}")

In [ ]:
# Create output directory
if model_type == "sngp":
    hf_model_dir = Path("./checkpoints/acevedo_sngp_resnet18/hf_checkpoint")
else:
    hf_model_dir = Path("./hf_checkpoint")

hf_model_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Created directory: {hf_model_dir}")

In [ ]:
# Save config.json
config_dict = {
    "architectures": [architectures],
    "model_type": model_type_str,
    "auto_map": {
        "AutoConfig": f"{hf_model_file}.SNGPConfig" if model_type == "sngp" else f"{hf_model_file}.BaselineClassifierConfig",
        "AutoModel": f"{hf_model_file}.{architectures}",
    },
    **model_config,
}

config_path = hf_model_dir / "config.json"
with open(config_path, "w") as f:
    json.dump(config_dict, f, indent=2)

print(f"✓ Saved config.json")

## Step 4: Save Model Code & Weights

In [ ]:
# Save weights
weights_path = hf_model_dir / "pytorch_model.bin"
torch.save(clean_state_dict, weights_path)

file_size_mb = weights_path.stat().st_size / 1e6
print(f"✓ Saved pytorch_model.bin ({file_size_mb:.1f} MB)")

In [ ]:
# Copy model code to checkpoint directory
if model_type == "sngp":
    hf_model_src = Path("src/hf_sngp_model.py")
else:
    hf_model_src = Path("src/hf_model.py")

hf_model_dst = hf_model_dir / hf_model_src.name
shutil.copy(hf_model_src, hf_model_dst)

print(f"✓ Copied {hf_model_src.name} to checkpoint directory")

## Step 5: Test Loading the Model Locally

In [ ]:
from transformers import AutoModel

# Load model (works WITHOUT src/ directory!)
model = AutoModel.from_pretrained(str(hf_model_dir), trust_remote_code=True)
model = model.to(device)
model.eval()

print(f"✓ Loaded model successfully!")
print(f"  Type: {type(model).__name__}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test inference
test_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    output = model(test_input)

print(f"✓ Inference successful!")
print(f"  Input: {test_input.shape}")
if isinstance(output, dict):
    for key in output.keys():
        if isinstance(output[key], torch.Tensor):
            print(f"  {key}: {output[key].shape}")
else:
    print(f"  Output: {output.shape}")

## Step 6: Verify Model Outputs Match Original

In [ ]:
# Load original model
if model_type == "sngp":
    from src.models.sngp_gpt import SNGPClassifier
    original_model = SNGPClassifier(num_classes=8, arch="resnet18", pretrained=False,
                                     rff_dim=1024, length_scale=1.0, ridge_penalty=1e-3,
                                     cov_momentum=0.999, mean_field=True, n_power_iterations_sn=1)
else:
    from src.models.baseline.baseline_models import BaselineClassifier
    original_model = BaselineClassifier(arch="resnet18", num_classes=8, dropout_p=0.5, pretrained=False)

original_model = original_model.to(device)
original_model.eval()

# Load weights
original_state_dict = checkpoint['state_dict']
remapped = {k.replace('net.', '', 1): v for k, v in original_state_dict.items()}
original_model.load_state_dict(remapped)

print(f"✓ Loaded original model")

In [ ]:
# Compare outputs
with torch.no_grad():
    if model_type == "sngp":
        original_out = original_model(test_input)[0]  # mean_field_logits
    else:
        original_out = original_model(test_input)
    
    hf_out = model(test_input)
    hf_out = hf_out['logits'] if isinstance(hf_out, dict) else hf_out

max_diff = (original_out - hf_out).abs().max().item()
is_match = torch.allclose(original_out, hf_out, atol=1e-5)

print(f"✓ Output comparison:")
print(f"  Max difference: {max_diff:.2e}")
print(f"  Match: {is_match}")
print(f"  ✓ Migration successful!" if is_match else "  ⚠ Check model structure")

## Step 7: (Optional) Push to HuggingFace Hub

In [ ]:
# Example: Push to HuggingFace Hub
"""
from huggingface_hub import upload_folder

repo_id = "your-username/your-model-name"
upload_folder(
    repo_id=repo_id,
    folder_path=str(hf_model_dir),
    repo_type="model",
)
print(f"✓ Model pushed to: https://huggingface.co/{repo_id}")
"""